# `AAWindowSampler.sample_benchmark_set()`

Run several named sampling **arms** in one call and stack them into a single benchmark `DataFrame`. This is a thin multi-arm orchestrator — each arm is one ordinary `sample_*` call in `'segments'` mode — so a downstream benchmark can consume any mix of negatives, unlabeled, and control rows uniformly.

In [ ]:
import aaanalysis as aa
import pandas as pd
aa.options["verbose"] = False

df_seq = pd.DataFrame({
    "entry":    ["P1", "P2", "P3"],
    "sequence": ["ACDEFGHIKLMNPQRSTVWY" * 2,
                 "YWVTSRQPNMLKIHGFEDCA" * 2,
                 "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ"],
    "pos":      [[5, 25], [15], []],
})
sampler = aa.AAWindowSampler(random_state=0)

## `arms`

`arms` is a mapping `{arm_name: {"method": <strategy>, **kwargs}}`. `method` is one of `'same_protein'`, `'different_protein'`, `'synthetic'`, `'motif_matched'` (`ut.LIST_STRATEGIES`); the remaining keys forward to the matching `sample_*` method. The reserved keys `df_seq`, `seed`, and `output_mode` are managed by the orchestrator and must not appear in an arm config.

Each arm keeps its own `role` and `strategy` tags; an extra `arm` column records which arm produced the row.

In [ ]:
arms = {
    "near_pos":  {"method": "same_protein",     "pos_col": "pos", "n": 4, "window_size": 5, "min_distance_to_pos": 2},
    "unlabeled": {"method": "different_protein", "pos_col": "pos", "n": 4, "window_size": 5},
    "control":   {"method": "synthetic",         "n": 3, "window_size": 5, "generator": "global_freq"},
}
df = sampler.sample_benchmark_set(df_seq=df_seq, arms=arms, seed=0)
aa.display_df(df=df[["arm", "entry_win", "role", "strategy", "window"]], show_shape=True)

The output is a row-wise concatenation of every arm's `'segments'` output (eight-column schema + `arm`). There is **no automatic cross-arm dedupe** — every sampled row is preserved; deduplicate protein-sourced windows on `entry_win` and synthetic windows on `window` if needed. `arm` + `strategy` + `role` together carry full row provenance.

## `seed`

`seed` is the master seed (falling back to the constructor `random_state`). Per-arm sub-seeds are derived deterministically with `numpy.random.SeedSequence`, so the same `seed` reproduces the same benchmark set:

In [ ]:
df_a = sampler.sample_benchmark_set(df_seq=df_seq, arms=arms, seed=42)
df_b = sampler.sample_benchmark_set(df_seq=df_seq, arms=arms, seed=42)
print("identical for equal seed:", df_a.equals(df_b))